# **0. Integrative Modeling**
---
<!-- <center>
    <img src="https://www.researchgate.net/publication/388724184/figure/fig1/AS:11431281312041874@1740539676158/Integrative-structure-modeling-workflow-A-Integrative-modeling-is-an-iterative-process.png" alt="IM workflow" style="display:block; margin:auto;"/>
</center> -->

<!-- [Bolinska and Sali (2025)](https://doi.org/10.1007/s11229-025-04911-0) -->

# **1.0 Integrative Modeling Platform (IMP)**
---
<!-- <center>
    <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSsBKg93jBkW9YU69Q8wOcO9Q6Ym_kXHMO-MA&s" alt="IMP logo" style="display:block; margin:auto;"/>
</center> -->

provides building blocks and tools to
- convert data from experiments into spatial restraints
- implement optimization and analysis techniques
- implement an integrative modeling procedure from scratch

written in C++

<hr>

# **2. Python Modeling Interface (PMI)**
---
High-level Python classes to simplify the construction of a modeling protocol in IMP

<hr>

# **3. Tutorial**
---

## 3.0 Integrative modeling of ADP-actin, gelsolin and tropomodulin

- Assume that we have a crystal structure of only the actin-gelsolin interface
- We would like to find the tropomodulin-actin-gelsolin complex structure from EM, crosslinking data and crystal structures

---

## 3.1 Set-up
---

- Installation instructions for google colab: https://integrativemodeling.org/2.24.0/doc/manual/installation.html

In [4]:
#@title 3.0.1 Installing IMP and dependencies
%%capture
!add-apt-repository -y ppa:salilab/ppa
!apt install imp

In [5]:
#@title 3.0.2 Setting up IMP and related packages
import sys, os, glob
sys.path.append(os.path.dirname(glob.glob('/usr/lib/python*/dist-packages/IMP')[0]))
os.environ['PYTHONPATH'] = os.path.dirname(glob.glob('/usr/lib/python*/dist-packages/IMP')[0])

In [6]:
#@title 3.0.3 Download and extract the tutorial content
#NOTE: Have to wait for some time even after the code cell is run to extract the zip
%%capture
if not os.path.exists("/content/main.zip"):
    !wget https://github.com/salilab/actin_tutorial/archive/main.zip
!rm -rf /content/actin_tutorial-main
!rm -rf /content/actin_tutorial-clean

# unzip files
os.chdir("/content/")
!unzip /content/main.zip

# create clean tutorial directory and copy necessary info only
!mkdir -p /content/actin_tutorial-clean/modeling/derived_data/xl
!mkdir -p /content/actin_tutorial-clean/modeling/derived_data/em
!mkdir -p /content/actin_tutorial-clean/modeling/gmm_files
!mkdir -p /content/actin_tutorial-clean/data/fasta
!mkdir -p /content/actin_tutorial-clean/data/pdb
!cp -r /content/actin_tutorial-main/data/fasta/* /content/actin_tutorial-clean/data/fasta
!cp /content/actin_tutorial-main/data/pdb/4pki.pdb /content/actin_tutorial-clean/data/pdb
!cp -r /content/actin_tutorial-main/modeling/derived_data/xl/* /content/actin_tutorial-clean/modeling/derived_data/xl
!cp /content/actin_tutorial-main/modeling/derived_data/em/4pki_20a.mrc /content/actin_tutorial-clean/modeling/derived_data/em/4pki_20a.mrc
#!cp /content/actin_tutorial-main/modeling/gmm_files/actin_gmm.mrc /content/actin_tutorial-clean/modeling/gmm_files
#!cp /content/actin_tutorial-main/modeling/gmm_files/gelsolin_gmm.mrc /content/actin_tutorial-clean/modeling/gmm_files
#!cp /content/actin_tutorial-main/modeling/gmm_files/actin_gmm.txt /content/actin_tutorial-clean/modeling/gmm_files
#!cp /content/actin_tutorial-main/modeling/gmm_files/gelsolin_gmm.txt /content/actin_tutorial-clean/modeling/gmm_files

In [7]:
#@title 3.0.4 Go to the tutorial directory
os.chdir("/content/actin_tutorial-clean/modeling/")
!pwd

/content/actin_tutorial-clean/modeling


# 3.1 Information gathering
---

See slides at: [IMP_Tutorial_NCBS_CCPEM_slides](https://docs.google.com/presentation/d/1hnKXZMIRj-xJZWRoe0UKNYyMbZcDkbO8/edit?usp=sharing&ouid=107573675581756580599&rtpof=true&sd=true)


In [8]:
#@title Preprocessing: Convert EM density map to GMM

num_gaussians = 20 # @param {type:"slider", min:10, max:20, step:5}

import subprocess
from string import Template
create_gmm_cmd = Template(
    "/usr/local/bin/python /usr/lib/python3.10/dist-packages/IMP/isd/create_gmm.py " +
    "${input_mrc} " +
    "${num_gaussians} " +
    "${output_txt} " +
    "-m ${output_mrc} "
)

create_gmm_ = create_gmm_cmd.substitute(
    input_mrc="/content/actin_tutorial-clean/modeling/derived_data/em/4pki_20a.mrc",
    num_gaussians=num_gaussians,
    output_txt=f"/content/actin_tutorial-clean/modeling/derived_data/em/4pki_20a_{num_gaussians}.txt",
    output_mrc=f"/content/actin_tutorial-clean/modeling/derived_data/em/4pki_20a_{num_gaussians}.mrc",
)

# print(create_gmm_)
result = subprocess.run(
    create_gmm_,
    shell=True,
    check=True,
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("GMM creation successful")
else:
    print("GMM creation failed")

GMM creation successful


# 3.2 System representation
---

See slides at: [IMP_Tutorial_NCBS_CCPEM_slides](https://docs.google.com/presentation/d/1hnKXZMIRj-xJZWRoe0UKNYyMbZcDkbO8/edit?slide=id.p12#slide=id.p12)

In [9]:
#@title 3.2.1 Topology file
import ipywidgets as widgets
from IPython.display import display, HTML

#@markdown - Topology file describes the system representation
#@markdown - You can include the following attributes:
#@markdown   - **molecule_name**: Name of the entity
#@markdown   - **color**: Color (name or hex-code) for the entity
#@markdown   - **fasta_id**: Fasta sequence header
#@markdown   - **fasta_fn**: Filename of the fasta file with sequence of the entity
#@markdown   - **pdb_fn**: Filename of the pdb file, if available, else `BEADS`
#@markdown   - **chain**: Chain id of the entity in the pdb file
#@markdown   - **residue_range**: Residue range of the entity (corresponding to the sequence)
#@markdown   - **pdb_offset**: Difference between PDB numbering and sequence numbering
#@markdown   - **bead_size**: Number of residues per bead
#@markdown   - **em_residues_per_gaussian**: Number of residues per gaussian for the EM restraint
#@markdown   - **rigid_body**: Index of the rigid body to which the entity belongs
#@markdown   - **super_rigid_body**: Index of the super rigid body to which the entity belongs

topology_txt = """
|directories|
|pdb_dir|../data/pdb/|
|fasta_dir|../data/fasta/|
|gmm_dir|./gmm_files/|

|topology_dictionary|
|molecule_name | color | fasta_fn     | fasta_id           | pdb_fn | chain | residue_range | pdb_offset | bead_size | em_residues_per_gaussian | rigid_body | super_rigid_body | chain_of_super_rigid_bodies |
|actin.0       |green  |4pkh.fasta.txt|actin               |4pki.pdb|A      |1,END          |0           |1          |10                        |1           |1                 |                             |
|geltrop.0     |red    |4pkh.fasta.txt|gelsolin-tropomyosin|4pki.pdb|G      |52,177         |-51         |1          |10                        |1           |1                 |                             |
|geltrop.0     |gray   |4pki.fasta.txt|gelsolin-tropomyosin|BEADS   |G      |178,195        |-51         |1          |10                        |1           |1                 |                             |
|geltrop.0     |blue   |4pkh.fasta.txt|gelsolin-tropomyosin|4pki.pdb|G      |1170,1349      |-1025       |1          |10                        |2           |1                 |                             |
"""

topology_file = "topology_file.txt" #@param {type:"string"}

display(HTML('''
<style>
    .mytext textarea {
        font-size: 0.7em; /* Change to your desired size */
        /* color: blue;     Optional: change text color */
    }
</style>
'''))

topology_input = widgets.Textarea(
    value=topology_txt,
    placeholder='Enter text here...',
    description='Topology:',
    layout=widgets.Layout(width='100%', height='250px')
)

topology_input.add_class("mytext")

# Create a button widget
button = widgets.Button(
    description='Save',
    button_style='success'
)

# Define the button click event
def show_save_topology(b):
    with open(topology_file, "w") as f:
        f.write(topology_input.value)
    print("Saved topology file to topology.txt")

button.on_click(show_save_topology)

# Display the widgets
display(topology_input, button)

Textarea(value='\n|directories|\n|pdb_dir|../data/pdb/|\n|fasta_dir|../data/fasta/|\n|gmm_dir|./gmm_files/|\n\…

Button(button_style='success', description='Save', style=ButtonStyle())

Saved topology file to topology.txt


In [10]:
#@title Create and set up model object from topology file

system_name = "actin_gelsolin_tropomyosin_complex" #@param{type: "string"}
force_create_gmm_files = False #param{type: "boolean"}

# Imports
from __future__ import print_function
import IMP
import IMP.pmi
import IMP.pmi.io
import IMP.pmi.io.crosslink
import IMP.pmi.topology
import IMP.pmi.macros
import IMP.pmi.restraints
import IMP.pmi.restraints.stereochemistry
import IMP.pmi.restraints.saxs
import IMP.pmi.restraints.crosslinking
import IMP.pmi.restraints.em
import IMP.pmi.dof
import ihm.cross_linkers
import IMP.atom
import IMP.saxs
import sys

# All IMP systems start out with a Model
mdl = IMP.Model()

# Read the topology file for a given state
t = IMP.pmi.topology.TopologyReader(topology_file)

# Create a BuildSystem macro to add a state from a topology file
bs = IMP.pmi.macros.BuildSystem(
    mdl,
    force_create_gmm_files=force_create_gmm_files,
    name=system_name,
)
bs.add_state(t)

# executing the macro will return the root hierarchy and degrees of freedom (dof) objects
root_hier, dof = bs.execute_macro()

# # It's useful to have a list of the molecules.
# molecules = t.get_components()

BuildSystem.add_state: setting up molecule actin copy number 0
BuildSystem.add_state: molecule actin sequence has 375 residues
BuildSystem.add_state: ---- setting up domain 0 of molecule actin
BuildSystem.add_state: -------- domain 0 of molecule actin extends from residue 1 to residue 375 
BuildSystem.add_state: -------- domain 0 of molecule actin represented by pdb file ../data/pdb/4pki.pdb 
BuildSystem.add_state: -------- domain 0 of molecule actin represented by gaussians 
BuildSystem.add_state: setting up molecule geltrop copy number 0
BuildSystem.add_state: molecule geltrop sequence has 324 residues
BuildSystem.add_state: ---- setting up domain 0 of molecule geltrop
BuildSystem.add_state: -------- domain 0 of molecule geltrop extends from residue 1 to residue 126 
BuildSystem.add_state: -------- domain 0 of molecule geltrop represented by pdb file ../data/pdb/4pki.pdb 
BuildSystem.add_state: -------- domain 0 of molecule geltrop represented by gaussians 
BuildSystem.add_state: ---

In [ ]:
#@title Show hierarchy

# IMP.atom.show_with_representations(root_hier)
skip_residue=True #@param{type: "boolean"}

def print_hierarchy(node, depth=0, step=2, skip_residue=True):
    print("-" * (depth * step) + node.get_name())
    for child in node.get_children():
        if skip_residue and "Residue" in child.get_name():
            continue
        print_hierarchy(child, depth + 1, step, skip_residue=skip_residue)

print_hierarchy(root_hier, depth=0, step=2, skip_residue=skip_residue)

actin_gelsolin_tropomyosin_complex
--State_0
----actin
------Frag_1-4
--------Frag_1-4: Res 1 Densities 1
----------1_bead
----------2_bead
----------3_bead
----------4_bead
------frags:5-72,74-375: Base
--------frags:5-72,74-375,: Res 1
------Frag_73-73
--------Frag_73-73: Res 1 Densities 1
----------73_bead
----geltrop
------frags:1-126: Base
--------frags:1-126,: Res 1
------Frag_127-144
--------Frag_127-144: Res 1 Densities 1
----------127_bead
----------128_bead
----------129_bead
----------130_bead
----------131_bead
----------132_bead
----------133_bead
----------134_bead
----------135_bead
----------136_bead
----------137_bead
----------138_bead
----------139_bead
----------140_bead
----------141_bead
----------142_bead
----------143_bead
----------144_bead
------frags:145-324: Base
--------frags:145-324,: Res 1


## Restraints

Restraints define functions that score the model based on
input information.

Restraint objects are first created in the definition.

To be evaluated, the restraint object must be added to the model using `add_to_model()`.

In some cases, sampled parameters for restraints must be added to the DOF
object

The `output_objects` list is used to collect all restraints
where we want to log the output in the STAT file.
Each restraint should be appended to this list.

### Connectivity restraint

Restrains residues/particles that are collected in sequence
This should be used for any system without an atomic force field
(e.g. CHARMM). We apply the restraint to each molecule.

In [11]:
#@title Code for Connectiviy restraint

output_objects = []

for m in root_hier.get_children()[0].get_children():
    cr = IMP.pmi.restraints.stereochemistry.ConnectivityRestraint(m)
    cr.add_to_model()
    output_objects.append(cr)

Adding sequence connectivity restraint between 1_bead  and  2_bead of distance 3.6
Adding sequence connectivity restraint between 2_bead  and  3_bead of distance 3.6
Adding sequence connectivity restraint between 3_bead  and  4_bead of distance 3.6
Adding sequence connectivity restraint between 4_bead  and  Residue_5 of distance 3.6
Adding sequence connectivity restraint between Residue_72  and  73_bead of distance 3.6
Adding sequence connectivity restraint between 73_bead  and  Residue_74 of distance 3.6
Adding sequence connectivity restraint between Residue_126  and  127_bead of distance 3.6
Adding sequence connectivity restraint between 127_bead  and  128_bead of distance 3.6
Adding sequence connectivity restraint between 128_bead  and  129_bead of distance 3.6
Adding sequence connectivity restraint between 129_bead  and  130_bead of distance 3.6
Adding sequence connectivity restraint between 130_bead  and  131_bead of distance 3.6
Adding sequence connectivity restraint between 131_

### Excluded volume restraint

Keeps particles from occupying the same area in space.
Here, we pass a list of both molecule chains to `included_objects` to
apply this to every residue.

We could also have passed root_hier to obtain the same behavior.

resolution=1000 applies this expensive restraint to the lowest resolution
for each particle.

In [12]:
#@title Code for Excluded volume restraint
evr = IMP.pmi.restraints.stereochemistry.ExcludedVolumeSphere(
    included_objects=[root_hier],
    resolution=1000
)

# Add all of the other restraints to the scoring function to start sampling
evr.add_to_model()
output_objects.append(evr)


### Crosslinking restraint

Restrains two particles via a distance restraint based on
an observed crosslink.

First, create the crosslinking database from the input file
The "standard keys" correspond to a crosslink csv file of the form:

<table>
<tr>
    <th>Protein1</th>
    <th>Residue1</th>
    <th>Protein2</th>
    <th>Residue2</th>
</tr>
<tr>
    <th>A</th>
    <th>18</th>
    <th>G</th>
    <th>24</th>
</tr>
<tr>
    <th>A</th>
    <th>18</th>
    <th>G</th>
    <th>146</th>
</tr>
<tr>
    <th>A</th>
    <th>50</th>
    <th>G</th>
    <th>146</th>
</tr>
<tr>
    <th>A</th>
    <th>50</th>
    <th>G</th>
    <th>171</th>
</tr>
<tr>
    <th>A</th>
    <th>50</th>
    <th>G</th>
    <th>189</th>
</tr>
</table>

This restraint allows for ambiguity in the crosslinked residues,
a confidence metric for each crosslink and multiple states.

See the PMI documentation or the MMB book chapter for a
full discussion of implementing crosslinking restraints.

This first step is used to translate the crosslinking data file.

The KeywordsConverter maps a column label from the xl data file
to the value that PMI understands.

In [13]:
#@title Code for crosslinking restraint
#@title Parameters

# Identify data files
xl_data = "./derived_data/xl/derived_xls.dat" #@param {type: "string"}

# Restraint weights
xl_weight = 10.0

xldbkc = IMP.pmi.io.crosslink.CrossLinkDataBaseKeywordsConverter()
# Here, we just use the standard keys.
xldbkc.set_standard_keys()
# One can define custom keywords using the syntax below.
# For example if the Protein1 column header is "prot_1"
# xldbkc["Protein1"]="prot_1"

# The CrossLinkDataBase translates and stores the crosslink information
# from the file "xl_data" using the KeywordsConverter.
xldb = IMP.pmi.io.crosslink.CrossLinkDataBase()
xldb.create_set_from_file(
    file_name=xl_data,
    converter=xldbkc
)

xlr = IMP.pmi.restraints.crosslinking.CrossLinkingMassSpectrometryRestraint(
    root_hier=root_hier,    # Must pass the root hierarchy to the system
    database=xldb,          # The crosslink database.
    length=25,              # The crosslinker plus side chain length
    resolution=1,           # The resolution at which to evaluate the crosslink
    slope=0.0001,           # This adds a linear term to the scoring function
                            #   to bias crosslinks towards each other
    weight=xl_weight,       # Scaling factor for the restraint score.
    linker=ihm.cross_linkers.dss)  # The linker chemistry

xlr.add_to_model()
output_objects.append(xlr)

gathering copies
defaultdict(<class 'int'>, {'actin': 0, 'geltrop': 0})
done pmi2 prelims
generating a new cross-link restraint
--------------
CrossLinkingMassSpectrometryRestraint: generating cross-link restraint between
CrossLinkingMassSpectrometryRestraint: residue 18 of chain actin and residue 24 of chain geltrop
CrossLinkingMassSpectrometryRestraint: with sigma1 SIGMA sigma2 SIGMA psi PSI
CrossLinkingMassSpectrometryRestraint: between particles Residue_18 and Residue_24

generating a new cross-link restraint
--------------
CrossLinkingMassSpectrometryRestraint: generating cross-link restraint between
CrossLinkingMassSpectrometryRestraint: residue 18 of chain actin and residue 146 of chain geltrop
CrossLinkingMassSpectrometryRestraint: with sigma1 SIGMA sigma2 SIGMA psi PSI
CrossLinkingMassSpectrometryRestraint: between particles Residue_18 and Residue_146

generating a new cross-link restraint
--------------
CrossLinkingMassSpectrometryRestraint: generating cross-link restraint be

### EM restraint

Scores a model based on its cross-correlation to an EM density.

Since cross-correlation is very expensive, we approximate both

the EM map and model as a set of 3D Gaussians (done in Representation).

### Steps

- Convert input density map to a GMM

- Select the particles on which to apply EM-restraint

- Define the EM-restraint
    - **densities**: Evaluate the restraint using these model densities
    - **target_fn**: The EM map, approximated as a gaussian mixture model (GMM)
    - **slope**: A small linear restraint to pull objects towards the EM map center
    - **scale_target_to_mass**: Normalizes the total density of the model wrs: EM map. Only set to true if the EM map and "densities" contain the same objects.
    - **weight**: The scaling factor for the EM score

- Add the restraint to the model

In [14]:
#@title Code for EM restraint

# Path where the gmm data is stored
gmm_data = "./derived_data/em/4pki_20a_20.txt" #@param {type: "string"}

# Restraint weights
em_weight = 1000.0 #@param {type: "number"}
em_slope = 0.00000001 #@param {type: "number"}
scale_target_to_mass = True #@param {type: "boolean"}

# First, collect all density particles from the model.
densities = IMP.atom.Selection(
    root_hier,
    representation_type=IMP.atom.DENSITIES
).get_selected_particles()

emr = IMP.pmi.restraints.em.GaussianEMRestraint(
    densities,
    target_fn=gmm_data,
    slope=em_slope,
    scale_target_to_mass=scale_target_to_mass,
    weight=em_weight
)

emr.add_to_model()
output_objects.append(emr)

will set target mass to 78082.25229999951
will scale target mass by 1.0
target num particles 20 total weight 78082.25229999953
model num particles 91 total weight 78082.25229999951
done EM setup


# Modeling

## Sampling

With our representation and scoring functions determined, we can now sample
the configurations of our model with respect to the information.

### Shuffling and optimization
First shuffle all particles to randomize the starting point of the
system. For larger systems, you may want to increase `max_translation`

Shuffling randomizes the bead positions. It's good to
allow these to optimize first to relax large connectivity
restraint scores.  100-500 steps is generally sufficient.

In [15]:
#@title Code for shuffling and optimization
IMP.pmi.tools.shuffle_configuration(
    root_hier,
    max_translation=50
)

dof.optimize_flexible_beads(500)

shuffling 2 rigid bodies
shuffling 23 flexible beads
optimize_flexible_beads: optimizing 23 flexible beads


### Replica exchange Monte Carlo sampling

In [16]:
#@title Parameters

test_mode = False #@param {type:"boolean"}

num_frames = 20 #@param {type:"integer"}

mc_steps= 10 #@param {type:"integer"}

best_scoring_models = 1 #@param {type:"integer"}


In [17]:
#@title Code for replica exchange Monte Carlo sampling

rex = IMP.pmi.macros.ReplicaExchange(
    mdl,
    # pass the root hierarchy
    root_hier=root_hier,
    # pass all objects to be moved ( almost always dof.get_movers() )
    monte_carlo_sample_objects=dof.get_movers(),
    # The output directory for this sampling run.
    global_output_directory='run1/output/',
    # Items in output_objects write information to the stat file.
    output_objects=output_objects,
    # Number of MC steps between writing frames
    monte_carlo_steps=mc_steps,
    # set >0 to store best PDB files (but this is slow)
    number_of_best_scoring_models= best_scoring_models,
    # Total number of frames to run / write to the RMF file.
    number_of_frames=num_frames,
    # Run in test mode (don't write anything)
    test_mode=test_mode)

# Ok, now we finally do the sampling!
rex.execute_macro()

# print(sr.evaluate())
print(emr.evaluate())
print(xlr.evaluate())

# Outputs are then analyzed in a separate analysis script.

Setting up MonteCarlo
Setting up ReplicaExchange
ReplicaExchange: Could not find MPI. Using Serial Replica Exchange
Setting up stat file
Setting up replica stat file
Setting up best pdb files
Setting up and writing initial rmf coordinate file
got existing rex object
Setting up production rmf files
ReplicaExchange: it generates initial.*.rmf3, stat.*.out, rmfs/*.rmf3 for each replica 
--- it stores the best scoring pdb models in pdbs/
--- the stat.*.out and rmfs/*.rmf3 are saved only at the lowest temperature
--- variables:
------ atomistic                      False
------ best_pdb_dir                   pdbs/
------ best_pdb_name_suffix           model
------ do_clean_first                 True
------ do_create_directories          True
------ geometries                     None
------ global_output_directory        run1/output/
------ initial_rmf_name_suffix        initial
------ mmcif                          False
------ molecular_dynamics_steps       10
------ monte_carlo_steps    

### Viweing output files

In [46]:
#@title View all the headers

stat_file = "./run1/output/stat.0.out" #@param{type: "string"}

import subprocess
from string import Template
process_output_cmd = Template(
    "/usr/local/bin/python " +
    "/usr/lib/python3.10/dist-packages/IMP/pmi/process_output.py " +
    "-f ${stat_file} " +
    "-p"
)

process_output_cmd = process_output_cmd.substitute(
    stat_file=stat_file,
)

print(process_output_cmd + "\n")

result = subprocess.run(
    process_output_cmd,
    shell=True,
    check=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print(result.stderr)

# !/usr/local/bin/python /usr/lib/python3.10/dist-packages/IMP/pmi/process_output.py -f /content/actin_tutorial-clean/modeling/run1/output/stat.0.out -p

/usr/local/bin/python /usr/lib/python3.10/dist-packages/IMP/pmi/process_output.py -f ./run1/output/stat.0.out -p

ConnectivityRestraint_Score
CrossLinkingMassSpectrometryRestraint_Data_Score
CrossLinkingMassSpectrometryRestraint_Distance_||0.1|actin|18|geltrop|24|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||1.1|actin|18|geltrop|146|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||10.1|actin|68|geltrop|203|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||11.1|actin|68|geltrop|272|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||12.1|actin|68|geltrop|289|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||13.1|actin|84|geltrop|146|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||14.1|actin|84|geltrop|202|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||15.1|actin|84|geltrop|203|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||16.1|actin|84|geltrop|230|0|PSI|
CrossLinkingMassSpectrometryRestraint_Distance_||17.1|actin|113|geltrop|28

In [82]:
#@title View header values from stat file

#@markdown **Headers**
#@markdown - **restraints**:
#@markdown   - `GaussianEMRestraint_None`
#@markdown   - `CrossLinkingMassSpectrometryRestraint_Score_`
#@markdown   - `ConnectivityRestraint_Score`
#@markdown   - `ExcludedVolumeSphere_Score`

#@markdown - **frames**:
#@markdown   - `rmf_frame_index`

#@markdown - **MCMC parameters**:
#@markdown   - `MonteCarlo_Acceptance_BallMover`
#@markdown   - `MonteCarlo_Acceptance_RigidBody`
#@markdown   - `MonteCarlo_Acceptance_Super`
#@markdown   - `ReplicaExchange_SwapSuccessRatio`

# header_name = "GaussianEMRestraint_None" #@param{type: "string"}
header_name = "rmf_frame_index" #@param["GaussianEMRestraint_None", "CrossLinkingMassSpectrometryRestraint_Score_", "ConnectivityRestraint_Score", "ExcludedVolumeSphere_Score", "rmf_frame_index", "MonteCarlo_Acceptance_BallMover", "MonteCarlo_Acceptance_RigidBody", "MonteCarlo_Acceptance_Super", "ReplicaExchange_SwapSuccessRatio"] {allow-input: true}
stat_file = "./run1/output/stat.0.out" #@param{type: "string"}

import subprocess
from string import Template
process_output_cmd = Template(
    "/usr/local/bin/python " +
    "/usr/lib/python3.10/dist-packages/IMP/pmi/process_output.py " +
    "-f ${stat_file} " +
    "-t ${header_name}"
)

process_output_cmd = process_output_cmd.substitute(
    stat_file=stat_file,
    header_name=header_name
)

print(process_output_cmd + "\n")

result = subprocess.run(
    process_output_cmd,
    shell=True,
    check=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print(result.stderr)


/usr/local/bin/python /usr/lib/python3.10/dist-packages/IMP/pmi/process_output.py -f ./run1/output/stat.0.out -t rmf_frame_index

rmf_frame_index 0
 
rmf_frame_index 1
 
rmf_frame_index 2
 
rmf_frame_index 3
 
rmf_frame_index 4
 
rmf_frame_index 5
 
rmf_frame_index 6
 
rmf_frame_index 7
 
rmf_frame_index 8
 
rmf_frame_index 9
 
rmf_frame_index 10
 
rmf_frame_index 11
 
rmf_frame_index 12
 
rmf_frame_index 13
 
rmf_frame_index 14
 
rmf_frame_index 15
 
rmf_frame_index 16
 
rmf_frame_index 17
 
rmf_frame_index 18
 
rmf_frame_index 19
 



In [84]:
#@title View values for multiple fields simultaneously

import ipywidgets as widgets
from IPython.display import display
import numpy as np

random_numbers = np.random.random((100))
options = [
    'Total_Score',
    'rmf_frame_index',
    'GaussianEMRestraint_None',
    'ConnectivityRestraint_Score',
    'ExcludedVolumeSphere_Score',
    'CrossLinkingMassSpectrometryRestraint_Data_Score',
    'ReplicaExchange_SwapSuccessRatio',
]
Dropdown_ = widgets.Dropdown(
    options=options,
    description='choose random number',
)
output = widgets.Output()

multiple_select = widgets.SelectMultiple(
    options=options,
    value=['rmf_frame_index'],
    description='Select',
    disabled=False,
    layout=widgets.Layout(width='50%', height='130px')
)

stat_file = "./run1/output/stat.0.out" #@param{type: "string"}

def on_change(change):

    import subprocess
    from string import Template
    process_output_cmd = Template(
        "/usr/local/bin/python " +
        "/usr/lib/python3.10/dist-packages/IMP/pmi/process_output.py " +
        "-f ${stat_file} " +
        "-s ${header_names}"
    )

    process_output_cmd = process_output_cmd.substitute(
        stat_file=stat_file,
        header_names=" ".join([str(x) for x in change['new']])
    )

    # print(process_output_cmd + "\n")

    result = subprocess.run(
        process_output_cmd,
        shell=True,
        check=True,
        capture_output=True,
        text=True
    )

    with output:
        output.clear_output()
        if result.returncode == 0:
            print(result.stdout)
        else:
            # output.clear_output()
            print(result.stderr)

multiple_select.observe(on_change, names='value')
display(multiple_select, output)
# !/usr/local/bin/python /usr/lib/python3.10/dist-packages/IMP/pmi/process_output.py -f /content/actin_tutorial-clean/modeling/run1/output/stat.0.out -s rmf_frame_index GaussianEMRestraint_None Total_Score

SelectMultiple(description='Select', index=(1,), layout=Layout(height='130px', width='50%'), options=('Total_S…

Output()

In [ ]:
#@title Extract the pre-computed tutorial results
!unzip "/content/actin_tutorial-main/modeling/run2.zip"

Archive:  /content/actin_tutorial-main/modeling/run2.zip
   creating: run2/
   creating: run2/output/
  inflating: run2/output/stat.3.out  
  inflating: run2/output/stat_replica.7.out  
  inflating: run2/output/stat.0.out  
  inflating: run2/output/stat_replica.6.out  
  inflating: run2/output/stat.4.out  
   creating: run2/output/pdbs/
  inflating: run2/output/initial.4.rmf3  
  inflating: run2/output/initial.2.rmf3  
  inflating: run2/output/stat.1.out  
  inflating: run2/output/stat_replica.4.out  
  inflating: run2/output/stat.6.out  
  inflating: run2/output/stat_replica.5.out  
  inflating: run2/output/initial.6.rmf3  
  inflating: run2/output/initial.1.rmf3  
  inflating: run2/output/stat_replica.0.out  
  inflating: run2/output/stat.2.out  
  inflating: run2/output/stat.5.out  
  inflating: run2/output/initial.0.rmf3  
  inflating: run2/output/initial.5.rmf3  
  inflating: run2/output/stat_replica.2.out  
  inflating: run2/output/stat_replica.1.out  
   creating: run2/output/rm

###TODO
Now you can view the stat files in
/content/actin_tutorial-clean/modeling/run2/output/

# Analysis

<center>
    <img
        src="https://integrativemodeling.org/tutorials/actin/analysis.png" alt="IMP logo"
        style="display:block; margin:auto;"
    />
</center>

# Summary

<center>
    <img src="https://integrativemodeling.org/tutorials/actin/four_stage.png" alt="IMP logo" style="display:block; margin:auto;"/>
</center>